In [1]:
!pip install -q transformers datasets accelerate openpyxl scikit-learn

In [2]:
import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

SEED = 42
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

ImportError: DLL load failed while importing missing: Uma política de Controle de Aplicativo bloqueou este arquivo.

In [ ]:
TRAIN_PATH = "../data/train.xlsx"
TEST_PATH = "../data/test1.xlsx"

DEV_MODEL_PATH = "../models/bertimbau_dev_best"
FINAL_MODEL_PATH = "../models/bertimbau_final_best"

OUTPUT_PATH = "../outputs/test1_rotulado.xlsx"

In [ ]:
print("Diretório atual:", os.getcwd())

Diretório atual: c:\Users\José\Documents\ep1-pln\notebooks


In [ ]:
df = pd.read_excel(TRAIN_PATH)

df["resp_text"] = df["resp_text"].astype(str)

print("Shape:", df.shape)
print()
print(df["clarity"].value_counts())

df.head()

Shape: (20092, 2)

clarity
c5      6892
c234    6853
c1      6347
Name: count, dtype: int64


,resp_text,clarity
0,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c5
1,"Prezada cidadã, As informações sobre óbitos ...",c1
2,"Prezado Senhor Julio, A Ouvidoria-Geral da P...",c1
3,"Prezado(a) Senhor(a), Esclarecemos que o Se...",c234
4,"Senhor, O Serviço de Informações ao Cidadão d...",c234


In [ ]:
df["text_group"] = pd.factorize(
    df["resp_text"]
)[0]

print("Textos únicos:", df["resp_text"].nunique())
print("Grupos criados:", df["text_group"].nunique())

Textos únicos: 18432
Grupos criados: 18432


In [ ]:
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

train_idx, val_idx = next(
    sgkf.split(
        X=df,
        y=df["clarity"],
        groups=df["text_group"]
    )
)

train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

print("Treino:", train_df.shape)
print("Validação:", val_df.shape)

print("\nDistribuição treino:")
print(train_df["clarity"].value_counts())

print("\nDistribuição validação:")
print(val_df["clarity"].value_counts())

Treino: (16073, 3)
Validação: (4019, 3)

Distribuição treino:
clarity
c5      5513
c234    5482
c1      5078
Name: count, dtype: int64

Distribuição validação:
clarity
c5      1379
c234    1371
c1      1269
Name: count, dtype: int64


In [ ]:
label2id = {
    "c1": 0,
    "c234": 1,
    "c5": 2
}

id2label = {
    0: "c1",
    1: "c234",
    2: "c5"
}

train_df["label"] = train_df["clarity"].map(label2id)
val_df["label"] = val_df["clarity"].map(label2id)

print(train_df[["clarity", "label"]].head())

  clarity  label
1      c1      0
2      c1      0
4    c234      1
5      c1      0
6    c234      1


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

MAX_LENGTH = 256

def tokenize_batch(batch):
    return tokenizer(
        batch["resp_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [ ]:
train_dataset = Dataset.from_pandas(
    train_df[["resp_text", "label"]].reset_index(drop=True)
)

val_dataset = Dataset.from_pandas(
    val_df[["resp_text", "label"]].reset_index(drop=True)
)

train_tokenized = train_dataset.map(
    tokenize_batch,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenize_batch,
    batched=True
)

print(train_tokenized)
print(val_tokenized)

Map:   0%|          | 0/16073 [00:00<?, ? examples/s]

Map:   0%|          | 0/4019 [00:00<?, ? examples/s]

Dataset({
    features: ['resp_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 16073
})
Dataset({
    features: ['resp_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4019
})


In [ ]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    preds = logits.argmax(axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(
            labels,
            preds,
            average="macro"
        )
    }

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

model = model.to(device)

print("Modelo carregado em:", device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Modelo carregado em: cuda


In [ ]:
training_args = TrainingArguments(
    output_dir=DEV_MODEL_PATH,

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=4,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),

    logging_steps=100,
    save_total_limit=1,

    report_to="none",

    seed=SEED
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,4.226137,1.037284,0.457079,0.449560
2,4.005470,1.022335,0.469769,0.469766
3,3.687578,1.049311,0.470266,0.469134


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3015, training_loss=3.998754154034515, metrics={'train_runtime': 783.3347, 'train_samples_per_second': 61.556, 'train_steps_per_second': 3.849, 'total_flos': 6077309444164728.0, 'train_loss': 3.998754154034515, 'epoch': 3.0})

In [ ]:
pred_output = trainer.predict(
    val_tokenized
)

pred_ids = pred_output.predictions.argmax(axis=-1)

print(
    classification_report(
        val_df["label"],
        pred_ids,
        target_names=["c1", "c234", "c5"]
    )
)

print("Matriz de confusão:")
print(
    confusion_matrix(
        val_df["label"],
        pred_ids
    )
)

              precision    recall  f1-score   support

          c1       0.49      0.53      0.51      1269
        c234       0.40      0.36      0.38      1371
          c5       0.52      0.53      0.52      1379

    accuracy                           0.47      4019
   macro avg       0.47      0.47      0.47      4019
weighted avg       0.47      0.47      0.47      4019

Matriz de confusão:
[[667 360 242]
 [438 496 437]
 [258 394 727]]


In [ ]:
trainer.save_model(
    DEV_MODEL_PATH
)

tokenizer.save_pretrained(
    DEV_MODEL_PATH
)

print("Modelo de desenvolvimento salvo em:", DEV_MODEL_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo de desenvolvimento salvo em: ../models/bertimbau_dev_best


In [ ]:
full_df = df.copy()

full_df["label"] = full_df["clarity"].map(label2id)

full_dataset = Dataset.from_pandas(
    full_df[["resp_text", "label"]].reset_index(drop=True)
)

full_tokenized = full_dataset.map(
    tokenize_batch,
    batched=True
)

print("Total de exemplos:", len(full_dataset))

Map:   0%|          | 0/20092 [00:00<?, ? examples/s]

Total de exemplos: 20092


In [ ]:
final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

final_model = final_model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

In [ ]:
final_training_args = TrainingArguments(
    output_dir=FINAL_MODEL_PATH,

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    num_train_epochs=3,

    weight_decay=0.01,

    save_strategy="epoch",
    save_total_limit=1,

    fp16=torch.cuda.is_available(),

    logging_steps=100,

    report_to="none",

    seed=SEED
)

In [ ]:
final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=full_tokenized,
    data_collator=data_collator
)

In [ ]:
final_trainer.train()

Step,Training Loss
100,4.409193
200,4.330229
300,4.327337
400,4.246638
500,4.246725
600,4.193180
700,4.213604
800,4.173983
900,4.214791
1000,4.212769


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3768, training_loss=3.8962703672437407, metrics={'train_runtime': 915.1788, 'train_samples_per_second': 65.863, 'train_steps_per_second': 4.117, 'total_flos': 7588676082639456.0, 'train_loss': 3.8962703672437407, 'epoch': 3.0})

In [ ]:
final_trainer.save_model(
    FINAL_MODEL_PATH
)

tokenizer.save_pretrained(
    FINAL_MODEL_PATH
)

print("Modelo final salvo em:", FINAL_MODEL_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo final salvo em: ../models/bertimbau_final_best


In [ ]:
test_df = pd.read_excel(TEST_PATH)

print("Shape teste:", test_df.shape)
print("Colunas:", test_df.columns.tolist())

test_df.head()

Shape teste: (900, 2)
Colunas: ['resp_text', 'clarity']


,resp_text,clarity
0,"Prezada Hellen, Dentre as pesquisas domici...",NaN
1,"Senhor Francisco, O Serviço de Informações a...",NaN
2,"Prezada Srª. Raylane, Em resposta a sua solic...",NaN
3,"Prezada Senhora, De acordo com o art.13 do D...",NaN
4,"Prezado(a) Senhor(a), Em atendimento ao ped...",NaN


In [ ]:
test_df["resp_text"] = test_df["resp_text"].astype(str)

test_dataset = Dataset.from_pandas(
    test_df[["resp_text"]].reset_index(drop=True)
)

test_tokenized = test_dataset.map(
    tokenize_batch,
    batched=True
)

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [ ]:
test_output = final_trainer.predict(
    test_tokenized
)

test_pred_ids = test_output.predictions.argmax(axis=-1)

test_pred_labels = np.array([
    id2label[i]
    for i in test_pred_ids
])

print("Número de previsões:", len(test_pred_labels))
print()
print(pd.Series(test_pred_labels).value_counts())

Número de previsões: 900

c5      379
c234    265
c1      256
Name: count, dtype: int64


In [ ]:
submission_df = test_df.copy()

submission_df["clarity"] = test_pred_labels

submission_df.head()

,resp_text,clarity
0,"Prezada Hellen, Dentre as pesquisas domici...",c234
1,"Senhor Francisco, O Serviço de Informações a...",c234
2,"Prezada Srª. Raylane, Em resposta a sua solic...",c5
3,"Prezada Senhora, De acordo com o art.13 do D...",c1
4,"Prezado(a) Senhor(a), Em atendimento ao ped...",c1


In [ ]:
assert len(submission_df) == 900
assert submission_df.shape[1] == 2

assert submission_df.columns.tolist() == [
    "resp_text",
    "clarity"
]

assert submission_df["clarity"].isna().sum() == 0

assert set(submission_df["clarity"].unique()).issubset(
    {"c1", "c234", "c5"}
)

assert (
    submission_df["resp_text"].values
    == test_df["resp_text"].values
).all()

print("Todas as validações passaram.")

Todas as validações passaram.


In [ ]:
submission_df.to_excel(
    OUTPUT_PATH,
    index=False
)

print("Arquivo gerado em:", OUTPUT_PATH)

Arquivo gerado em: ../outputs/test1_rotulado.xlsx


In [ ]:
check_df = pd.read_excel(
    OUTPUT_PATH
)

print("Shape:", check_df.shape)
print("Colunas:", check_df.columns.tolist())
print()
print(check_df["clarity"].value_counts())
print()
print("Nulos:", check_df["clarity"].isna().sum())

Shape: (900, 2)
Colunas: ['resp_text', 'clarity']

clarity
c5      379
c234    265
c1      256
Name: count, dtype: int64

Nulos: 0
